In [34]:
import pandas as pd
from pathlib import Path

HI_ACCOUNT_DATA_PATH = Path().cwd() / "Data-HI" / "HI-Small_accounts.csv"
HI_TRANSACTIONS_DATA_PATH = Path().cwd() / "Data-HI" / "HI-Small_Trans.csv"

accounts_df = pd.read_csv(HI_ACCOUNT_DATA_PATH)
trans_df = pd.read_csv(HI_TRANSACTIONS_DATA_PATH)

In [35]:
trans_df['Timestamp'] = pd.to_datetime(trans_df['Timestamp'])

trans_df = trans_df.rename(columns={'Account':'From Account'})
trans_df = trans_df.rename(columns={'Account.1':'To Account'})

# trans_df['From Bank'] = trans_df['From Bank'].astype('category')
# trans_df['To Bank'] = trans_df['To Bank'].astype('category')
# trans_df['Payment Currency'] = trans_df['Payment Currency'].astype('category')
# trans_df['Receiving Currency'] = trans_df['Receiving Currency'].astype('category')
# trans_df['Payment Format'] = trans_df['Payment Format'].astype('category')

In [36]:
trans_df.dtypes

Timestamp             datetime64[us]
From Bank                      int64
From Account                     str
To Bank                        int64
To Account                       str
Amount Received              float64
Receiving Currency               str
Amount Paid                  float64
Payment Currency                 str
Payment Format                   str
Is Laundering                  int64
dtype: object

In [37]:
accounts_df = accounts_df.rename(columns={'Account Number': 'Account'})

# accounts_df['Bank Name'] = accounts_df['Bank Name'].astype('category')
# accounts_df['Bank ID'] = accounts_df['Bank ID'].astype('category')

In [38]:
accounts_df.dtypes

Bank Name        str
Bank ID        int64
Account          str
Entity ID        str
Entity Name      str
dtype: object

In [39]:
total_lines = len(accounts_df)
unique_accounts = accounts_df['Account'].nunique()

print(total_lines)
print(unique_accounts)

duplicates_amt = total_lines - unique_accounts
print(duplicates_amt)

518581
518573
8


In [40]:
accounts_df[accounts_df.duplicated(subset=['Account'], keep=False)].sort_values(by='Account')

,Bank Name,Bank ID,Account,Entity ID,Entity Name
62794,Australia Bank #44,28248,80A7FD400,800D28430,Partnership #36506
122088,Australia Bank #47,27755,80A7FD400,800CCD520,Corporation #33736
61324,Australia Bank #44,28248,80A7FDE00,800D15C50,Corporation #34165
124530,Australia Bank #47,27755,80A7FDE00,800D28430,Partnership #36506
160206,Germany Bank #908,138832,80FA55EF0,8009067F0,Corporation #49502
409360,Italy Bank #97,13858,80FA55EF0,8007F7670,Corporation #22542
149001,Germany Bank #908,138832,80FA56340,8006CF910,Partnership #28810
417905,Italy Bank #97,13858,80FA56340,80087BC70,Corporation #25811
216212,Italy Bank #95,1490,81211BA20,80078A8D0,Corporation #24600
505755,Germany Bank #945,142574,81211BA20,8007D4530,Partnership #27156


In [41]:
accounts_df['Universal_Account_ID'] = accounts_df['Bank ID'].astype(str) + "_" + accounts_df['Account'].astype(str)

trans_df['From_Universal_ID'] = trans_df['From Bank'].astype(str) + "_" + trans_df['From Account'].astype(str)

trans_df['To_Universal_ID'] = trans_df['To Bank'].astype(str) + "_" + trans_df['To Account'].astype(str)

print(len(accounts_df))
print(accounts_df['Universal_Account_ID'].nunique())

518581
518581


In [42]:
amostra_contas_ids = accounts_df['Universal_Account_ID'].sample(n=500, random_state=42)

accounts_sample = accounts_df[accounts_df['Universal_Account_ID'].isin(amostra_contas_ids)].copy()

trans_sample = trans_df[trans_df['From_Universal_ID'].isin(amostra_contas_ids)].copy()

print(f"Contas na amostra: {accounts_sample.shape[0]}")
print(f"Transações na amostra: {trans_sample.shape[0]}")

Contas na amostra: 500
Transações na amostra: 4671


In [46]:
import featuretools as ft

accounts_sample = accounts_sample.reset_index(drop=True)
trans_sample = trans_sample.reset_index(drop=True)

if 'transaction_id' in trans_sample.columns:
    trans_sample = trans_sample.drop(columns=['transaction_id'])
trans_sample['transaction_id'] = trans_sample.index

logical_types_acc = {
    'Bank Name': 'Categorical',
    'Bank ID': 'Categorical',
    'Account': 'Categorical',
    'Entity ID': 'Categorical',
    'Entity Name': 'Categorical',
    'Universal_Account_ID': 'Categorical'
}

logical_types_trans = {
    'From Bank': 'Categorical',
    'To Bank': 'Categorical',
    'From Account': 'Categorical',
    'To Account': 'Categorical',
    'Receiving Currency': 'Categorical',
    'Payment Currency': 'Categorical',
    'Payment Format': 'Categorical',
    'From_Universal_ID': 'Categorical',
    'To_Universal_ID': 'Categorical'
}

es = ft.EntitySet(id='aml_data')

es = es.add_dataframe(
    dataframe_name='accounts',
    dataframe=accounts_sample,
    index='Universal_Account_ID',
    logical_types=logical_types_acc
)

es = es.add_dataframe(
    dataframe_name='transactions',
    dataframe=trans_sample,
    index='transaction_id',
    time_index='Timestamp',
    logical_types=logical_types_trans
)

relationship = ft.Relationship(
    entityset=es,
    parent_dataframe_name="accounts",
    parent_column_name="Universal_Account_ID",
    child_dataframe_name="transactions",
    child_column_name="From_Universal_ID"
)
es = es.add_relationship(relationship)

print("\nEntitySet construído com sucesso absoluto!")
print(es)

WoodworkNotInitError: Woodwork not initialized for this DataFrame. Initialize by calling DataFrame.ww.init